
# 05 实盘运维与逐笔对账

> **本章要解决的问题**：回测赚钱，实盘却对不上账：是滑点、费用，还是漏记了一笔成交？账户突然断开、订单被拒、保证金告急时，先做哪一步？事故之后，怎么证明“我们已经处理过并且没有留下新坑”？

## 本章导读

策略上线只是开始，真正的风险在**运维**：事件是否记全、账是否平、状态是否一致、变更是否可回滚。这一章按“事件 → 账 → 状态 → 变更”展开：

1. 实盘运维的四个对象：事件、账、状态、变更；
2. 逐笔事件模型：订单、成交、费用、现金；
3. 事件库：SQLite 与幂等写入；
4. 每日对账：持仓、现金、费用，以及差异定位；
5. 监控、告警与 kill switch；
6. 断线重连、拒单、强平；
7. 事故 runbook 与版本回滚；
8. 上线前的运维检查清单。

### 学习目标

- 能设计一份逐笔事件 schema，并用回放算出当日持仓、现金、费用和权益。
- 能用 SQLite 建事件表，演示幂等写入，并从库里重算账。
- 能说清内部、柜台、交易所三方对账的顺序，以及差异怎么分类定位。
- 能写至少 5 条监控规则，并把告警分级映射到 kill switch 的状态转移。
- 能写一份断线、拒单、强平的处置 runbook。
- 能设计版本清单与回滚步骤，并开出可复盘的事故单。
- 能列出实盘上线前必须逐条打勾的运维清单。

### 先修

第 12、13、14、15 章的 WT 多因子、回放、工具链与失败复盘；会 Python 的 `json`、`sqlite3` 与 pandas 基础。

---

### 环境准备



In [1]:

import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("matplotlib 中文字体已设置")


matplotlib 中文字体已设置



## 在量化研究中的位置

- **谁用**：交易员、策略开发、中台对账、风控、运维值班。
- **输入**：订单/成交/费用事件、柜台与交易所回报、持仓和资金快照、版本与配置。
- **输出**：当日对账单、差异清单、告警与处置记录、事故单、回滚记录。
- **在流程的哪一步**：从“策略能跑”到“账户敢放钱”的最后一公里，也是每天收盘后的固定动作。
- **失败模式**：事件漏记、重复投递、快照覆盖事件、告警没有动作、回滚没有验证、事故没有复盘。
- **最小工作流**：采集事件 → 落库 → 回放算账 → 三方对账 → 告警/处置 → 复盘归档。

---

## 1. 实盘运维管什么：事件、账、状态、变更

### 1.1 四个对象

| 对象 | 内容 | 出问题时的表现 |
|---|---|---|
| 事件 | 订单、成交、撤单、拒单、费用、出入金 | 回放结果与账户不一致 |
| 账 | 持仓、现金、费用、权益 | 对不上、越对越乱 |
| 状态 | 连接、报单开关、kill switch、持仓限额 | 该停时没停、该恢复时恢复错 |
| 变更 | 策略版本、配置、参数、部署时间 | 说不清“现在跑的是哪一版” |

### 1.2 逐笔事件是源，快照是派生

很多事故来自同一个坏习惯：**用“每日快照”覆盖明细**。快照只能用于核对，不能替代事件。事件是账的“源数据”，快照是从事件算出来的“视图”；视图错了可以重算，源错了就永远对不上。

### 1.3 最小工作流

采集事件 → 落库（唯一键去重）→ 回放算持仓/现金/费用 → 与柜台、交易所对账 → 有差异进事故流程 → 收盘归档。

---

## 2. 逐笔事件：一份最小 schema

### 2.1 字段

| 字段 | 例子 | 作用 |
|---|---|---|
| `seq` | 1, 2, 3… | 单调序号，幂等与排序 |
| `ts` | 2026-11-02 09:00:01 | 事件时间 |
| `type` | ORDER_NEW / FILL / FEE / CASH_MOVE | 事件类型 |
| `order_id` | O01 | 订单主键 |
| `symbol` | RB | 品种/合约 |
| `side` | BUY / SELL | 方向 |
| `qty` | 10 | 数量（手） |
| `price` | 3020.0 | 成交价 |
| `fee` | 6.04 | 手续费 |

### 2.2 幂等与顺序

网络重试、断线补发、消息队列重复投递，都会让同一条事件到达两次。处理方式不是“祈祷不重复”，而是给每条事件一个唯一键（`seq` 或 `client_order_id + fill_id`），入库时用 `INSERT OR IGNORE` 或 upsert 去重。**幂等不是优化，是正确性要求。**

### 2.3 代码：JSON 事件与回放



In [2]:

import json
from collections import defaultdict

MULT = {"RB": 10, "M": 10, "AG": 15}      # 教学示例合约乘数
LAST = {"RB": 3020.0, "M": 2800.0, "AG": 7200.0}


def ev(seq, clock, typ, **kw):
    row = {"seq": seq, "ts": "2026-11-02 " + clock, "type": typ}
    row.update(kw)
    return row


events = [
    ev(1, "08:55:00", "CASH_MOVE", amount=1000000.0, ref="INIT"),
    ev(2, "09:00:01", "ORDER_NEW", order_id="O01", symbol="RB", side="BUY", qty=10, price=3020.0),
    ev(3, "09:00:01", "ORDER_ACK", order_id="O01"),
    ev(4, "09:00:02", "FILL", order_id="O01", symbol="RB", side="BUY", qty=10, price=3020.0, fee=6.04),
    ev(5, "09:05:00", "ORDER_NEW", order_id="O02", symbol="M", side="SELL", qty=5, price=2800.0),
    ev(6, "09:05:00", "ORDER_ACK", order_id="O02"),
    ev(7, "09:05:03", "FILL", order_id="O02", symbol="M", side="SELL", qty=5, price=2800.0, fee=2.80),
    ev(8, "09:10:00", "ORDER_NEW", order_id="O03", symbol="AG", side="BUY", qty=2, price=7200.0),
    ev(9, "09:10:01", "ORDER_REJECT", order_id="O03", reason="可用资金不足"),
    ev(10, "09:20:00", "ORDER_NEW", order_id="O04", symbol="AG", side="BUY", qty=1, price=7200.0),
    ev(11, "09:20:01", "ORDER_ACK", order_id="O04"),
    ev(12, "09:20:04", "FILL", order_id="O04", symbol="AG", side="BUY", qty=1, price=7200.0, fee=1.80),
    ev(13, "14:50:00", "ORDER_NEW", order_id="O05", symbol="RB", side="SELL", qty=4, price=3030.0),
    ev(14, "14:50:01", "ORDER_ACK", order_id="O05"),
    ev(15, "14:50:02", "FILL", order_id="O05", symbol="RB", side="SELL", qty=4, price=3030.0, fee=2.42),
]


def replay(events):
    cash, fees, pos = 0.0, 0.0, defaultdict(int)
    for e in events:
        if e["type"] == "CASH_MOVE":
            cash += e["amount"]
        elif e["type"] == "FILL":
            sign = 1 if e["side"] == "BUY" else -1
            cash -= sign * e["qty"] * e["price"] * MULT[e["symbol"]]
            cash -= e["fee"]
            fees += e["fee"]
            pos[e["symbol"]] += sign * e["qty"]
    return {"cash": round(cash, 2), "fees": round(fees, 2), "positions": dict(pos)}


book = replay(events)
mv = sum(q * LAST[s] * MULT[s] for s, q in book["positions"].items())
equity = book["cash"] + mv
print("现金", round(book["cash"], 2), "费用", round(book["fees"], 2))
print("持仓", book["positions"], "市值", round(mv, 2), "权益", round(equity, 2))
print("当日净盈亏", round(equity - 1000000.0, 2))

blob = json.dumps(events, ensure_ascii=False, indent=1)
loaded = json.loads(blob)
print("JSON 往返", len(events), len(loaded), "一致", loaded == events, "字节", len(blob.encode("utf-8")))


现金 851186.94 费用 13.06
持仓 {'RB': 6, 'M': -5, 'AG': 1} 市值 149200.0 权益 1000386.94
当日净盈亏 386.94
JSON 往返 15 15 一致 True 字节 2095



**读数**：回放得到现金 851,186.94、费用 13.06；持仓 RB 6 手、M −5 手、AG 1 手，按最新价折市值 149,200，权益 1,000,386.94，当日净盈亏 +386.94。其中 400 元来自 RB 的 10 手买、4 手卖（价差 10 元 × 4 手 × 10 吨），其余持仓按最新价盯市为 0；再扣 13.06 手续费，就是 +386.94。JSON 序列化再反序列化后 15 条事件完全一致，共 2,095 字节。

**常见错误**：

1. 用“成交价 × 手数”当现金流，忘了乘合约乘数。一手 RB 是 10 吨，价格差 1 元就是一车 10 元的账。
2. 用快照覆盖事件。快照对不上时无据可查，事件对不上还能重放。
3. 把被拒订单也计入成交。`ORDER_REJECT` 只说明报单失败，不产生现金和持仓变动。

**停下来想一想**：为什么说“事件回放”比“每天存一份持仓快照”更可靠？

<details><summary>对照你的思路</summary>

快照只保存结果，不保存过程。一旦快照本身错了（漏了一笔成交、时间点错、口径变），你无法知道错在哪里，只能人工翻录像。事件流保存了“发生了什么”的顺序，任何一天的账都可以从任意起点重放出来；如果两份事件源不一致，还能逐条比对定位。快照适合做校验点和回滚点，但不应成为唯一的账本。

</details>

---

## 3. 事件库：SQLite 与幂等写入

### 3.1 表结构

最小表只需要四列：`seq`（主键）、`ts`、`type`、`payload`（JSON 文本）。把变化字段塞进 payload，schema 稳定、扩展方便；需要查询的字段再拆列或建索引。

### 3.2 幂等写入

`INSERT OR IGNORE` 依赖主键冲突。断线补发、重复消费、手工补录都可能触发同一条事件，唯一键保证“写两次等于写一次”。写库和回放要能互相验证：**从库里重算出来的现金和费用，必须等于内存回放的结果**，否则说明落库有丢失或错序。

### 3.3 代码：落库、重放与重算



In [3]:

import json
import sqlite3

con = sqlite3.connect(":memory:")
con.execute("""
CREATE TABLE events (
    seq     INTEGER PRIMARY KEY,
    ts      TEXT NOT NULL,
    type    TEXT NOT NULL,
    payload TEXT NOT NULL
)
""")
rows = []
for e in loaded:
    payload = {k: v for k, v in e.items() if k not in ("seq", "ts", "type")}
    rows.append((e["seq"], e["ts"], e["type"], json.dumps(payload, ensure_ascii=False)))
con.executemany("INSERT OR IGNORE INTO events VALUES (?, ?, ?, ?)", rows)
con.executemany("INSERT OR IGNORE INTO events VALUES (?, ?, ?, ?)", rows)  # 重试/重放
con.commit()
print("事件行数", con.execute("SELECT COUNT(*) FROM events").fetchone()[0])
print("类型分布", con.execute("SELECT type, COUNT(*) FROM events GROUP BY type ORDER BY type").fetchall())

fill_rows = con.execute("SELECT payload FROM events WHERE type = 'FILL' ORDER BY seq").fetchall()
cash_db, fee_db = 0.0, 0.0
for (payload,) in fill_rows:
    f = json.loads(payload)
    sign = 1 if f["side"] == "BUY" else -1
    cash_db -= sign * f["qty"] * f["price"] * MULT[f["symbol"]] + f["fee"]
    fee_db += f["fee"]
cash_db += sum(e["amount"] for e in loaded if e["type"] == "CASH_MOVE")
print("由 SQLite 重算现金/费用", round(cash_db, 2), round(fee_db, 2))
print("与内存回放一致", abs(cash_db - book["cash"]) < 1e-9 and abs(fee_db - book["fees"]) < 1e-9)
con.close()


事件行数 15
类型分布 [('CASH_MOVE', 1), ('FILL', 4), ('ORDER_ACK', 4), ('ORDER_NEW', 5), ('ORDER_REJECT', 1)]
由 SQLite 重算现金/费用 851186.94 13.06
与内存回放一致 True



**读数**：15 条事件入库；即使把同一批事件重复写两遍，行数仍是 15，这就是幂等。类型分布为 CASH_MOVE 1、FILL 4、ORDER_ACK 4、ORDER_NEW 5、ORDER_REJECT 1——被拒的 O03 只有 NEW 和 REJECT，没有 FILL，正好和上节的回放对上。从 SQLite 重算的现金 851,186.94、费用 13.06 与内存回放完全一致。

**常见错误**：

1. 用自增 id 当幂等键。重试会生成新 id，重复入账；应该用业务唯一键。
2. 先写库再校验，校验失败就手工改库。正确做法是保留原始事件，用“重放”修视图。
3. 事件表允许更新历史行。审计上应只追加（append-only），修正用冲正事件。

**停下来想一想**：如果交易所回报和柜台成交对不上，应该改哪一份数据？

<details><summary>对照你的思路</summary>

都不应该“直接改”。先保留两份原始记录，各自落库；再按“交易所成交编号 / 柜台成交编号 / 本地订单号”三方匹配，找出差异类型（漏报、重复、价格错、数量错、时点错）。确认后才用冲正事件修正内部账，并把差异和处理写进事故单。直接改历史事件等于毁掉审计线索。

</details>

---

## 4. 每日对账：持仓、现金、费用

### 4.1 三方对账

| 来源 | 看什么 | 特点 |
|---|---|---|
| 内部事件回放 | 持仓、现金、费用、权益 | 最细，可能有漏记 |
| 柜台 | 成交、持仓、资金 | 券商口径，可能有延迟 |
| 交易所结算 | 结算价、手续费、保证金 | 权威，但 T+1 才有 |

顺序：先对**总额**（权益、现金），再对**分项**（费用、持仓），最后对**逐笔**（成交明细）。反过来做会陷入细节，越对越乱。

### 4.2 差异分类

| 差异类型 | 典型原因 | 处理 |
|---|---|---|
| 资金/费用 | 手续费口径、结算时间、出入金 | 对账单逐项核 |
| 成交/持仓 | 漏记、重复、部分成交 | 按成交编号逐笔匹配 |
| 价格/数量 | 乘数错、单位错、价格错 | 回到原始回报核对 |
| 时点 | 跨日、夜盘归属、结算时点 | 统一时区与交易日历 |

### 4.3 代码：差异清单与定位



In [4]:

import pandas as pd

internal = replay(loaded)
statement = {
    "cash": internal["cash"] - 12.5,
    "fees": internal["fees"] + 12.5,
    "positions": {"RB": 7, "M": -5, "AG": 0},
}

diffs = []
for field, label in [("cash", "现金"), ("fees", "费用")]:
    d = round(statement[field] - internal[field], 2)
    if abs(d) > 1e-9:
        diffs.append([label, internal[field], statement[field], d, "资金/费用"])
for sym in sorted(set(internal["positions"]) | set(statement["positions"])):
    a = internal["positions"].get(sym, 0)
    b = statement["positions"].get(sym, 0)
    if a != b:
        diffs.append([f"持仓 {sym}", a, b, b - a, "成交/持仓"])
diff_df = pd.DataFrame(diffs, columns=["项目", "内部", "券商", "差异", "类别"])
print(diff_df.to_string(index=False))

cash_gap = round(statement["cash"] - internal["cash"], 2)
fee_gap = round(statement["fees"] - internal["fees"], 2)
print("现金差 = -费用差 ?", abs(cash_gap + fee_gap) < 1e-9)
pos_gap_equity = sum((statement["positions"].get(s, 0) - internal["positions"].get(s, 0)) * LAST[s] * MULT[s]
                     for s in set(internal["positions"]) | set(statement["positions"]))
print("持仓差按最新价折成权益", round(pos_gap_equity, 2), "权益差", round(cash_gap + pos_gap_equity, 2))


   项目        内部        券商    差异    类别
   现金 851186.94 851174.44 -12.5 资金/费用
   费用     13.06     25.56  12.5 资金/费用
持仓 AG      1.00      0.00  -1.0 成交/持仓
持仓 RB      6.00      7.00   1.0 成交/持仓
现金差 = -费用差 ? True
持仓差按最新价折成权益 -77800.0 权益差 -77812.5



**读数**：差异表四行：现金差 −12.5、费用差 +12.5（互为相反数，典型的“费用记多/记少”）；持仓 AG 内部 1、券商 0（内部比券商多一张多单），RB 内部 6、券商 7（内部漏记了一笔买入）。关键在最后一行：持仓差按最新价折成权益是 −77,800，加上现金差 −12.5，权益总差 −77,812.5。**如果只看权益总额，会被持仓差放大几十倍，根本定位不到 12.5 元的手续费。**

**常见错误**：

1. 先比权益总额。总额差是多个小差叠加的结果，先把现金、费用、持仓分项拆开。
2. 把“现金差=费用差相反数”当巧合。多数情况下就是费用口径或结算时点问题。
3. 发现持仓差就直接手工改持仓。应该回到成交明细，找漏记/重复的那一笔。

**停下来想一想**：为什么现金差和费用差刚好互为相反数时，优先怀疑费用而不是成交？

<details><summary>对照你的思路</summary>

成交会同时改变现金和持仓，且金额通常远大于手续费，不会只造成 12.5 元这种小额镜像差。现金减少、费用等额增加，正好是“多记了一笔费用”的特征。当然还要用对账单确认：如果费用金额、时间、收费项目都对不上，再往结算时间、最低收费等方向查。

</details>

---

## 5. 监控、告警与 kill switch

### 5.1 指标分层

- 行情：断档、延迟、跳价；
- 报单：拒绝率、往返延迟、超时；
- 成交：成交率、部分成交、滑点；
- 持仓：库存、限额、集中度；
- 资金：保证金占用、风险度、当日亏损；
- 系统：CPU、内存、磁盘、进程存活。

### 5.2 阈值要带窗口和持续条件

单点越界多半是噪声：一笔大单让“延迟 p99”跳到 2 秒，并不代表系统坏了。规则应写成“指标 + 窗口 + 持续时长 + 阈值”，例如“5 分钟内拒单率 > 2% 且持续 3 个采样点”。

### 5.3 告警分级与动作

| 级别 | 例子 | 动作 |
|---|---|---|
| CRITICAL | 当日亏损超限、持仓对不平、保证金告急 | 触发 kill switch：停新单、撤挂单、减仓、通知值班 |
| HIGH | 拒单率上升、行情断档、延迟升高 | 告警 + 人工确认，必要时降速 |
| MED | 单笔滑点偏大、成交率下降 | 记录 + 观察，不打断交易 |

### 5.4 代码：规则评估与状态机



In [5]:

import pandas as pd

rules = pd.DataFrame([
    {"id": "R01", "name": "拒单率", "metric": "reject_rate", "op": ">", "threshold": 0.02, "severity": "HIGH"},
    {"id": "R02", "name": "下单延迟 p99(ms)", "metric": "latency_ms", "op": ">", "threshold": 800, "severity": "MED"},
    {"id": "R03", "name": "当日亏损比例", "metric": "loss_pct", "op": ">", "threshold": 0.03, "severity": "CRITICAL"},
    {"id": "R04", "name": "持仓对账不平", "metric": "pos_mismatch", "op": ">", "threshold": 0, "severity": "CRITICAL"},
    {"id": "R05", "name": "行情断档(秒)", "metric": "feed_gap", "op": ">", "threshold": 30, "severity": "HIGH"},
])
metrics = {"reject_rate": 0.031, "latency_ms": 620, "loss_pct": 0.034, "pos_mismatch": 1, "feed_gap": 12}


def evaluate(rules, metrics):
    fired = []
    for r in rules.itertuples():
        value = metrics.get(r.metric)
        if value is None:
            continue
        hit = value > r.threshold if r.op == ">" else value < r.threshold
        if hit:
            fired.append({"id": r.id, "name": r.name, "value": value,
                          "threshold": r.threshold, "severity": r.severity})
    order = {"CRITICAL": 0, "HIGH": 1, "MED": 2}
    return sorted(fired, key=lambda x: order[x["severity"]])


alerts = evaluate(rules, metrics)
for a in alerts:
    print(a["severity"], a["id"], a["name"], "值", a["value"], "阈值", a["threshold"])

state, history = "NORMAL", [("09:30:00", "NORMAL", "开盘初始化")]
for a in alerts:
    if a["severity"] == "CRITICAL" and state != "HALT":
        state = "HALT"
        history.append(("10:02:11", state, f'kill switch: {a["name"]}'))
    elif a["severity"] == "HIGH" and state == "NORMAL":
        state = "WARN"
        history.append(("10:02:11", state, f'告警: {a["name"]}'))
print("最终状态", state)
for h in history:
    print(h)
print("HALT 动作", ["停止新单", "撤全部挂单", "减仓到限额", "通知值班", "记录事故单"])


CRITICAL R03 当日亏损比例 值 0.034 阈值 0.03
CRITICAL R04 持仓对账不平 值 1 阈值 0.0
HIGH R01 拒单率 值 0.031 阈值 0.02
最终状态 HALT
('09:30:00', 'NORMAL', '开盘初始化')
('10:02:11', 'HALT', 'kill switch: 当日亏损比例')
HALT 动作 ['停止新单', '撤全部挂单', '减仓到限额', '通知值班', '记录事故单']



**读数**：5 条规则里有 3 条命中：R03（当日亏损 3.4% > 3%）和 R04（持仓对不平 1 > 0）都是 CRITICAL，R01（拒单率 3.1% > 2%）是 HIGH；R02 的 620 ms 没到 800、R05 的 12 秒没到 30 秒，不误报。状态机从 NORMAL 走到 HALT，只记录第一次转入；动作清单是“停新单、撤挂单、减仓到限额、通知值班、记录事故单”。

**常见错误**：

1. 只有告警没有动作。告警响了没人处理，等于没有告警。
2. 阈值没有窗口。单点噪声触发 kill switch，把正常交易打断。
3. kill switch 只“停新单”不“撤挂单”。已有挂单仍可能成交，风险还在。
4. 恢复条件不明确。停容易、恢复难；恢复必须人工确认并留痕。

**停下来想一想**：kill switch 触发后，应该自动全部平仓吗？

<details><summary>对照你的思路</summary>

不一定。自动全平在流动性差、行情断档时会放大亏损，甚至打出极端价格；更稳妥的是分级：先停新单、撤挂单、禁止加仓，再根据风险度、流动性、价差决定减仓节奏，由值班人员确认。只有在“保证金告急、强平在即”这类极端场景才允许自动平仓，而且要有明确的执行上限与回滚预案。

</details>

---

## 6. 断线重连、拒单、强平

### 6.1 断线重连

- 靠心跳和顺序号判断链路是否活着；
- 重连后**先查在途订单和成交**，再恢复报单，避免“以为没成交，其实成交了”；
- 用 `client_order_id` 做幂等，重复下单不会变成两笔仓位；
- 行情源和交易链路分开恢复：行情没恢复前，不要用旧价格报价。

### 6.2 拒单

常见原因：可用资金不足、超出持仓限额、合约停牌/涨跌停、合约到期、权限不足、价格超出报单区间。处理原则：**不要对同一笔单循环重试**，先查原因，再决定改价、改量或放弃。

### 6.3 强平

强平是“别人的动作”，不是策略的止损。风险度接近追保线时，主动减仓/补资金/降杠杆，比等强平更可控；强平通常伴随流动性最差、价差最大的时刻。

---

## 7. 事故 runbook 与版本回滚

### 7.1 runbook 表

| 字段 | 内容 |
|---|---|
| 场景 | 行情断线、拒单、持仓不平、保证金不足、版本异常 |
| 识别信号 | 哪个指标、什么阈值、持续多久 |
| 第一动作 | 先做什么（切源、停新单、核对在途） |
| 禁止动作 | 不要盲目重发、不要改事件库、不要边查边改生产 |
| 升级路径 | 值班 → 交易员 → 风控 → 负责人 |

### 7.2 版本清单

每次部署至少记录：当前版本、上一版本、提交号、配置 sha、部署时间、部署人、变更内容。没有这份清单，“回滚”只能靠记忆。

### 7.3 回滚步骤

确认事故与部署相关 → 停止发布 → 回滚策略版本 → 恢复配置并核对 sha → 重放当日事件核对持仓/现金 → 记录事故单与复盘结论。

### 7.4 代码：runbook、断线处置与回滚



In [6]:

import pandas as pd

runbook = pd.DataFrame([
    ["行情断线", "feed_gap>30 秒或心跳丢失", "切备用源，只撤单不报单", "不要盲目重发订单", "值班→负责人"],
    ["下单拒单", "拒单率>2% 持续 5 分钟", "查资金/持仓/涨跌停/合约状态", "不要循环重试同一单", "交易员→风控"],
    ["持仓不平", "对账 diff≠0", "冻结新开仓，核对成交明细", "不要手工改事件库", "中台→负责人"],
    ["保证金不足", "风险度>80%", "减仓/补资金/降杠杆", "不要等强平", "风控→负责人"],
    ["版本异常", "部署后错误率上升", "回滚到上一版本", "不要边查边改生产", "发布人→负责人"],
], columns=["场景", "识别信号", "第一动作", "禁止动作", "升级路径"])
incident = {"id": "INC-20261102-01", "type": "行情断线", "start": "10:15:03",
            "end": "10:15:18", "impact": "0 笔成交，2 笔撤单"}
row = runbook.set_index("场景").loc[incident["type"]]
print("事故", incident["id"], "→", incident["type"])
print("第一动作", row["第一动作"], "| 禁止", row["禁止动作"])
steps = ["10:15:03 心跳丢失，触发告警",
         "10:15:08 切备用行情源，暂停新报单",
         "10:15:12 与柜台核对在途订单与成交",
         "10:15:18 恢复报单，仅允许撤单"]
for s in steps:
    print(" ", s)

manifest = {"current": "v1.3.2", "previous": "v1.3.1",
            "deployed_at": "2026-11-02 09:35:00", "config_sha": "a1b2c3d4"}
rollback = [
    "确认事故与本次部署相关（时间窗 < 30 分钟）",
    f"回滚策略版本 {manifest['current']} → {manifest['previous']}",
    "恢复上一版配置文件，并核对 sha",
    "重放当日事件到回滚后版本，核对持仓/现金",
    "记录事故单与复盘结论",
]
print("版本清单", manifest)
for i, s in enumerate(rollback, 1):
    print(i, s)


事故 INC-20261102-01 → 行情断线
第一动作 切备用源，只撤单不报单 | 禁止 不要盲目重发订单
  10:15:03 心跳丢失，触发告警
  10:15:08 切备用行情源，暂停新报单
  10:15:12 与柜台核对在途订单与成交
  10:15:18 恢复报单，仅允许撤单
版本清单 {'current': 'v1.3.2', 'previous': 'v1.3.1', 'deployed_at': '2026-11-02 09:35:00', 'config_sha': 'a1b2c3d4'}
1 确认事故与本次部署相关（时间窗 < 30 分钟）
2 回滚策略版本 v1.3.2 → v1.3.1
3 恢复上一版配置文件，并核对 sha
4 重放当日事件到回滚后版本，核对持仓/现金
5 记录事故单与复盘结论



**读数**：事故单 `INC-20261102-01` 是行情断线 15 秒，影响“0 笔成交、2 笔撤单”；runbook 给出第一动作“切备用源，只撤单不报单”，明确禁止“盲目重发订单”。回滚清单 5 步，最后一步是“重放当日事件核对持仓/现金”——**回滚不是把版本换回去就完了，必须用账验证。**

**常见错误**：

1. 事故单只写“已恢复”，不写影响面、时间线、根因。
2. 回滚后不重放对账。新旧版本的事件处理逻辑可能不同，账未必自动平。
3. 把 runbook 写成口号。每条必须有可执行的信号、动作、禁止项和升级人。

**停下来想一想**：为什么事故处理里要专门写“禁止动作”？

<details><summary>对照你的思路</summary>

人在压力下会本能地“多试几次”：重发订单、重启进程、手工改库。这些动作往往把一次可恢复的小故障放大成重复下单、账实不符、数据污染的二次事故。“禁止动作”把最容易犯的错提前写死，让值班者先停下来、按流程处置。

</details>

---

## 8. 上线前检查清单

### 8.1 数据与账

- [ ] 事件有唯一键，重复投递不会重复入账；
- [ ] 事件表 append-only，修正用冲正事件；
- [ ] 每日回放结果与柜台/交易所对账；
- [ ] 对账差异有分类、有责任人、有时限。

### 8.2 状态与监控

- [ ] kill switch 有明确触发条件与动作；
- [ ] 告警有分级、有值班、有恢复条件；
- [ ] 断线重连有“先查在途再报单”的流程；
- [ ] 拒单、强平有预案，不靠临场发挥。

### 8.3 变更与复盘

- [ ] 版本、配置、部署时间有清单；
- [ ] 回滚步骤演练过，且回滚后对账；
- [ ] 事故单包含时间线、影响、根因、改进项；
- [ ] 每次事故的改进项有跟踪闭环。

---

## 本章小结

### 一张表

| 环节 | 做法 | 防什么 |
|---|---|---|
| 事件 | 唯一键 + append-only | 漏记、重复、不可追溯 |
| 回放 | 从事件重算持仓/现金/费用 | 快照错误 |
| 对账 | 总额 → 分项 → 逐笔 | 差异越对越乱 |
| 告警 | 指标 + 窗口 + 分级 | 误报、漏报 |
| kill switch | 状态机 + 明确动作 | 该停没停 |
| runbook | 信号/动作/禁止/升级 | 临场慌乱 |
| 版本 | 清单 + 回滚 + 重放验证 | 回滚不彻底 |

### 八句话

1. 事件是源，快照是视图；视图可以重算，源错了就没救。
2. 幂等不是优化，是正确性要求。
3. 对账先总额后分项，先现金后持仓。
4. 现金差与费用差互为相反数，先查费用口径。
5. 告警必须有动作，kill switch 必须有恢复条件。
6. 断线重连先查在途订单，再恢复报单。
7. 强平是别人的动作，别等它替你止损。
8. 回滚后用账验证，而不是用“感觉”验证。

---

## 本章习题

**1.** 一天的事件流里，`ORDER_REJECT` 会不会影响持仓和现金？为什么？

**2.** 用 10 手 RB（乘数 10 吨/手）在 3020 买入、3030 卖出 4 手，剩余 6 手按 3020 盯市。忽略费用，求这份持仓的已实现盈亏和当日总盈亏。

**3.** 为什么事件表要用业务唯一键，而不是自增主键做幂等？

**4.** 内部账现金 100 万、券商 100 万零 13；内部费用 13.06、券商 25.56。先怀疑什么？下一步查什么？

**5.** 对账发现持仓 AG 内部 1、券商 0，按最新价 7200、乘数 15 折算，权益差多少？为什么不能只看权益总额？

**6.** 写一条合格的告警规则，至少包含哪四个要素？举一个“单笔延迟高但不应触发 kill switch”的例子。

**7.** 断线重连后，为什么不能立刻按原策略继续报单？应该先做什么？

**8.** 回滚版本后，为什么还要“重放当日事件”？如果重放后账仍不平，怎么处理？

<details><summary>参考答案</summary>

**1.** 不会。拒单只表示报单未被接受，没有成交、没有现金变动、没有持仓变动；它只影响“报单成功率”这类监控指标。

**2.** RB 买 10 手、卖 4 手：已实现盈亏 $(3030-3020)\times4\times10=400$ 元；剩余 6 手按 3020 盯市，浮动盈亏 0；当日总盈亏 = 400 元（忽略费用）。费用另扣 13.06 后为 386.94。

**3.** 自增主键由数据库生成，网络重试、重复消费会生成新的 id，导致同一条事件重复入账；业务唯一键（seq、成交编号、client_order_id）能识别“同一条事件”，配合 `INSERT OR IGNORE` 实现幂等。

**4.** 先怀疑费用：现金差 −12.5、费用差 +12.5 互为相反数，典型是多记了 12.5 元费用。下一步查对账单的收费项目、最低收费、结算时间，再确认是否有漏记成交（成交不会只差 12.5）。

**5.** 权益差 $=(0-1)\times7200\times15=-108{,}000$ 元。权益总额里混着现金和所有持仓，单个持仓差会被价格和乘数放大；只盯总额无法区分“小费用差”和“大持仓差”，必须分项对账。

**6.** 四要素：指标（如“5 分钟拒单率”）、窗口（5 分钟）、阈值（>2%）、持续条件（连续 3 个采样点）。例子：单笔报单延迟 2 秒但整体 p99 正常、无拒单，说明是个别网络抖动，记录观察即可，不应触发 kill switch。

**7.** 因为断线期间可能已有成交、挂单状态变化，本地状态不可信。应先与柜台核对在途订单和成交，把本地事件补全、对平持仓与现金，再恢复报单；恢复初期可只允许撤单、限制新单。

**8.** 回滚只换了代码和配置，事件处理逻辑可能不同，账未必自动一致；重放是为了验证回滚后的版本能正确重算当天的持仓/现金/费用。若仍不平，先冻结新开仓，保留原始事件与两份结果，按差异类型逐笔定位，必要时用冲正事件修正，并升级为事故处理。

</details>

---

## 参考资料

- WonderTrader-plus / wtpy-plus 的回放与账户语义（第 12–15 章）。
- 交易所与期货公司结算单、成交回报字段说明（以各自官方规则为准）。
- Kleppmann, M. *Designing Data-Intensive Applications*（事件、幂等、日志）。
- Google SRE Book（监控、告警、事故响应与复盘）。
- 第 12–15 章的 WT 回放与账户语义；advanced 01《事件驱动回测与撮合》的撮合与账本细节。
